# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library for programmatic access to datasets described by a Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
def print_metadata(ds):
    md = ds.metadata
    print(f"Name: {md.name}\n")
    print(f"Identifier: {md.identifier}")
    print(f"Description: {md.description}\n")
    print(f"Published: {md.datePublished}")
    print(f"Version: {md.version}")
    print(f"License: {md.license}")
    print(f"Keywords: {', '.join(md.keywords)}\n")
    print(f"Data Collection: {md.dataCollection}")
    print(f"Data Use cases: {md.dataUseCases}")

print_metadata(dataset)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we list all record sets and for each their available fields and columns, referencing them by their Croissant `@id`.

> **Note:** Croissant datasets may define one or more record sets (data tables). You must use the right `@id` when referencing fields.

In [ ]:
# List all record sets and their fields/columns by @id
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the Croissant schema. Is the data listed as a resource or DataFileObject?")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs.name} (@id: {rs.id})\n  Description: {getattr(rs, 'description', None)}")
        print("  Fields:")
        for field in getattr(rs, 'field', []):
            print(f"    - {field.name} (@id: {field.id}, type: {field.dataType})")
        print("")

# If recordSet is empty, we should look for DataFileObject(s):
if not record_sets:
    # Many biological data packages use distribution to store files, so 
    # explore those (often these are the data tables)
    distributions = getattr(dataset.metadata, 'distribution', [])
    for d in distributions:
        print(f"Distribution: @id: {d.id}, name: {getattr(d, 'name', None)}, url: {getattr(d, 'contentUrl', None)}")

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis. Use the record set and field `@id`s discovered above.

If record sets are not available but distributions are, use their `@id` or URL.

In [ ]:
# ---
# Identify the record set @id (or use distribution @id if recordSet is empty).
# We'll auto-discover the recordSet @ids or fallback to DataFileObject/distribution.

# Get record set(s) @id
record_set_ids = [rs.id for rs in getattr(dataset.metadata, 'recordSet', [])] if getattr(dataset.metadata, 'recordSet', None) else []
# If not available, fall back to DataFileObjects via distribution
if not record_set_ids:
    record_set_ids = [d.id for d in getattr(dataset.metadata, 'distribution', [])]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records from: {record_set_id}")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# Show the columns of the first record set loaded
if dataframes:
    sample_record_set_id = list(dataframes.keys())[0]
    print(f"Available columns for {sample_record_set_id}:\n", dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head(5))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll demonstrate this using a numeric field from the dataset. Please consult the columns printed above to select a suitable field.

In [ ]:
# Choose the first loaded record set
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # Try to find a numeric column (commonly named 'Total Peptide Count', or 'MW', etc, for this proteomics dataset)
    # Adjust names as needed. We'll attempt to auto-detect a float/integer column.
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None and len(df) > 0:
        # Try to coerce columns to numeric and pick first that works
        for col in df.columns:
            conv = pd.to_numeric(df[col], errors='coerce')
            if conv.notnull().sum() > 0:
                numeric_field = col
                df[col] = conv
                break
    if numeric_field:
        print(f"Using numeric field: '{numeric_field}' for EDA.")

        # Remove NaNs for filtering
        valid_vals = df[numeric_field].dropna()
        if len(valid_vals) == 0:
            print("No valid values found for the chosen numeric field.")
        else:
            # Choose an arbitrary threshold (e.g., > median)
            threshold = valid_vals.median()
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records where {numeric_field} > {threshold} (kept {len(filtered_df)} rows).\n")
            print(filtered_df.head())

            # Normalization
            if filtered_df[numeric_field].std() > 0:
                filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
                print(f"\nNormalized {numeric_field} (z-score normalization):")
                print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
            else:
                print(f"All entries have the same value for {numeric_field}; skipping normalization.")

            # Try grouping by a categorical field (e.g., 'Sample' or similar)
            group_field = None
            for col in df.columns:
                if col != numeric_field and df[col].dtype == object and df[col].nunique() < 20:
                    group_field = col
                    break
            if group_field:
                grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
                print(f"\nGrouped data mean by '{group_field}':")
                print(grouped_df)
            else:
                print("No categorical group field found for grouping.")
    else:
        print("No numeric field detected for EDA.")
else:
    print("No dataframes loaded to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

E.g., plot a histogram of the numeric variable filtered above or a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    if 'filtered_df' in locals() and len(filtered_df) > 0 and numeric_field is not None:
        plt.figure(figsize=(8, 5))
        sns.histplot(filtered_df[numeric_field].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field} (filtered)")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()
        # If normalized field exists
        if f"{numeric_field}_normalized" in filtered_df.columns:
            plt.figure(figsize=(8, 5))
            sns.boxplot(x=filtered_df[f"{numeric_field}_normalized"])
            plt.title(f"Boxplot of normalized {numeric_field} (filtered)")
            plt.xlabel(f"{numeric_field}_normalized")
            plt.show()
    else:
        print("No filtered numeric data available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using the Croissant schema, explored its record sets/fields by `@id`, extracted the data into pandas DataFrames, and performed essential exploratory data analysis with filtering, normalization, and grouping.

The dataset covers mass spectrometry-based protein quantitation in human extracellular vesicles. We recommend consulting the Croissant schema metadata for authoritative `@id` referencing and for deeper integration into ML workflows.

For advanced use, adjust filtering/grouping to specific columns and match your scientific questions.